# Data download
## Binance spot · historical OHLCV · circulating market cap · research diagnostics

**Research period: 1 January 2021 through the last completed UTC day. No strategy, signals, portfolio construction or backtest.**

This is a standalone, auditable acquisition notebook. It discovers the **published historical spot archive**, including pairs not present in today's trading list. It preserves raw archives, validates publisher checksums, exports familiar pair tickers, and produces twelve explanatory figures. It is not a claim of exchange-complete or point-in-time institutional coverage: those require independent reconciliation and identity review.

### Important boundary: Binance is not the entire crypto market

Binance listings alone cannot reproduce a broad CoinMarketCap sample of illiquid tokens. Historical archive discovery reduces current-list selection bias; it does **not** eliminate missing archives, symbol reuse, survivorship or exchange-selection bias. Do not interpret downloaded instruments as a verified historically investable universe.

### Quick start

1. Use Python 3.10+ with `numpy`, `pandas>=2.2`, `matplotlib>=3.9` and Jupyter. The downloader uses Python's standard-library HTTP client; no Binance key or trading permissions are needed.
2. Run the notebook with the default `RUN_DOWNLOAD=False` to load the program and run offline tests. The included live-validation examples are explicitly bounded samples, not the full universe.
3. Set `RUN_DOWNLOAD=True` and run all cells to acquire **all discovered spot pairs**, with daily candles from 2021. Change `QUOTES` to `["USDT"]` only if you deliberately want a narrower dataset. Whole-archive acquisition may take hours; inspect storage capacity first.
4. Review `Coin_Prices/asset_mapping.csv`. Ten common identities are seeded; other symbols are deliberately not guessed. The free market-cap source has limited coverage. For broader history, use the CoinGecko adapter with a plan entitled to 2021 data and reviewed IDs.
5. Inspect `run_status.json`, `failures.csv`, `quality_report.csv`, and `market_cap_coverage.csv` before passing any data to later research.

Install in your own environment if needed: `python -m pip install 'numpy>=1.26' 'pandas>=2.2' 'matplotlib>=3.9' jupyter`.

## 1 · A precise acquisition contract

| Decision | Default / interpretation |
|---|---|
| Venue and market | Binance global **spot**, published monthly and daily archives; not Binance.US or futures |
| Instruments | All archive symbols, all quotes; no top-30 cap, present-day rank or current `TRADING` filter |
| Time | UTC, start inclusive / end exclusive; only completed bars |
| Frequency | `1d`; `1h` also validated. Other intervals intentionally fail validation |
| Monthly versus daily | Monthly file when published; daily files only for months without a monthly file |
| OHLC units | Quote currency per base coin; `BTC-USDT` is not `BTC-USD` |
| Volume | Base volume and quote turnover, plus trade count and taker-buy volumes |
| Market cap | Separate provider observation, in USD; never Binance price × today's supply |
| Missing observations | Remain missing. No forward/back fill, fabricated OHLC, or delisting returns |
| Token classes | Raw acquisition retains stablecoins, wrappers and leveraged spot products if archived; no substring exclusions |
| Historical identities | Pair-name inference is flagged. Migrations/rebrands require date-bounded manual identity review |
| Reproducibility | Immutable content-addressed raw bytes, per-run output partitions, SHA-256 hashes, configuration and request receipts |

Data are source-revised histories retrieved now, not evidence of what a researcher could actually have known in 2021. `retrieved_at` records acquisition time, **not** first publication time. Missing order books, funding, borrow costs and historical trading filters are outside this OHLCV project.

In [1]:
# Configuration: daily, all historical spot instruments, from 2021.
from pathlib import Path
import pandas as pd

RUN_DOWNLOAD = False  # Set True deliberately: the full archive is a large download.
RUN_SELF_TESTS = True

CONFIG = {
    "ROOT": "Coin_Prices",
    "START": "2021-01-01",
    "END_EXCLUSIVE": pd.Timestamp.now(tz="UTC").normalize().strftime("%Y-%m-%d"),
    "INTERVAL": "1d",  # supported: 1d or 1h; daily recommended for first run
    "QUOTES": None,  # None = ALL archive quotes; ["USDT"] narrows acquisition
    "SYMBOLS": None,  # None = all; ["BTCUSDT", "ETHUSDT"] is a smoke test
    "FETCH_CURRENT_METADATA": False,  # Optional exchangeInfo annotation, never a universe filter
    "PAIR_OVERRIDES": {},  # e.g. {"BTCUSDT": ["BTC", "USDT"]}; review retired symbols
    "REQUEST_PAUSE_SECONDS": 0.05,  # serial requests, bounded retries, respects Retry-After
    "RECHECK_ARCHIVES": False,  # False pins cached versions; True checks publisher revisions
    "CAP_SOURCE": "coinmetrics",  # free but LIMITED; alternative: "coingecko_pro"
    "REFRESH_CAPS": True,  # new raw versions retained; False reuses pinned responses
    "SUGGEST_MARKET_CAP_IDS": False,  # optional CG symbol candidates; never auto-approved
    "MAKE_PLOTS": True,
    "SHOW_PLOTS": True,
    "RESEARCH_QUOTE": "USDT",  # plots use one quote; never add BTC/EUR/USDT turnover
}


## 2 · Transport, integrity and recovery

Requests are serial and bounded rather than launching hundreds of uncontrolled API calls. HTTP 401/403/418/451 stops the run; it does not switch endpoints to route around restrictions. A provider-requested pause above 60 seconds stops with a rerun-later message. No credentials are written to output files.

Raw download recovery is automatic: a rerun reuses verified files and rebuilds output into a new run directory. This avoids partially updated datasets. A writer lock prevents concurrent runs against the same root. After a killed process, inspect whether a process remains active before manually removing a stale `writer.lock`.

Checksums establish byte integrity, **not** economic accuracy. Monthly and daily archive publishing can lag. Missing, invalid or overlapping source rows are reported, not quietly repaired.

In [2]:
"""Research-only Binance archive acquisition; no signals, trades, or backtests."""
from __future__ import annotations
import os, io, json, time, hashlib, zipfile, tempfile, re, platform
import urllib.request, urllib.error, urllib.parse
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ssl
try:
    import certifi
    SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())
except Exception:
    SSL_CONTEXT = ssl.create_default_context()

VERSION = "1.0.0"
ARCHIVE = "https://data.binance.vision/"
BUCKET = "https://s3-ap-northeast-1.amazonaws.com/data.binance.vision"
CG = "https://pro-api.coingecko.com/api/v3"
CM = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/"
COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
        "quote_volume", "trades", "taker_buy_volume", "taker_buy_quote_volume", "ignore"]
KNOWN_QUOTES = sorted({"USDT", "USDC", "FDUSD", "BUSD", "TUSD", "USDP", "DAI", "USD1", "U",
    "BTC", "ETH", "BNB", "EUR", "GBP", "AUD", "BRL", "TRY", "RUB", "UAH", "NGN",
    "BIDR", "IDRT", "ZAR", "PLN", "RON", "ARS", "MXN", "JPY", "AEUR", "EURI",
    "BKRW", "PAX", "USDS", "USDSB", "VAI", "UST", "USTC", "BVND"}, key=lambda x: (-len(x), x))
CAP_COLUMNS = ["asset_id", "ticker", "timestamp", "market_cap_usd", "reference_price_usd",
               "global_volume_24h_usd", "circulating_supply", "provider", "provider_id", "retrieved_at"]

def utcnow():
    return pd.Timestamp.now(tz="UTC")

def sha256(data):
    return hashlib.sha256(data).hexdigest()

def atomic_bytes(path, data):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=".partial-", dir=path.parent)
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary): os.unlink(temporary)

def write_json(path, obj):
    atomic_bytes(path, json.dumps(obj, indent=2, default=str, allow_nan=False).encode())

def write_csv(path, frame):
    # CSV.gz is deliberately portable: pandas alone can read all data products.
    import gzip
    raw = frame.to_csv(index=False).encode()
    atomic_bytes(path, gzip.compress(raw, mtime=0) if str(path).endswith(".gz") else raw)

class AccessBlocked(RuntimeError): pass

class HTTP:
    """Serial, rate-limited requests; credentials are headers, never in logs."""
    def __init__(self, root, pause=0.25, attempts=6, max_bytes=256_000_000):
        self.root = Path(root); self.pause = pause; self.attempts = attempts
        self.max_bytes = max_bytes; self.last_call = 0
        self.audit = []

    def get(self, url, params=None, headers=None, missing_ok=False):
        split = urllib.parse.urlsplit(url)
        url = urllib.parse.urlunsplit((split.scheme, split.netloc,
            urllib.parse.quote(split.path, safe="/%:@"), split.query, split.fragment))
        if params: url += "?" + urllib.parse.urlencode(params)
        last_exc = None
        for attempt in range(self.attempts):
            time.sleep(max(0, self.pause - (time.monotonic() - self.last_call)))
            self.last_call = time.monotonic()
            try:
                request = urllib.request.Request(url, headers={"User-Agent": "CryptoResearchDownload/1.0", **(headers or {})})
                with urllib.request.urlopen(request, timeout=45, context=SSL_CONTEXT) as r:
                    data = r.read(self.max_bytes + 1)
                    if len(data) > self.max_bytes: raise ValueError("Response exceeds configured byte limit")
                    self.audit.append({"url": url, "at": str(utcnow()), "status": r.status,
                                       "bytes": len(data), "sha256": sha256(data)})
                    return data
            except urllib.error.HTTPError as exc:
                last_exc = exc
                self.audit.append({"url": url, "at": str(utcnow()), "status": exc.code})
                if exc.code == 404 and missing_ok: return None
                if exc.code in (401, 403, 418, 451):
                    raise AccessBlocked(f"HTTP {exc.code}: access/entitlement blocked at {urllib.parse.urlsplit(url).netloc}; stop, do not route around it") from None
                if exc.code not in (429, 500, 502, 503, 504): raise
                retry_after = exc.headers.get("Retry-After", "")
                try: delay = float(retry_after)
                except ValueError:
                    try: delay = (parsedate_to_datetime(retry_after) - datetime.now(timezone.utc)).total_seconds()
                    except (ValueError, TypeError): delay = 2 ** (attempt + 1)
                if delay > 60: raise RuntimeError(f"Provider requests a {delay:.0f}s pause; rerun later") from None
            except (urllib.error.URLError, TimeoutError, ConnectionError, OSError) as exc:
                last_exc = exc
                delay = 2 ** (attempt + 1)
            if attempt == self.attempts - 1:
                detail = f": {last_exc}" if last_exc else ""
                raise RuntimeError(f"Retry budget exhausted for {url}{detail}") from last_exc
            time.sleep(max(0, delay))

    def cached(self, url, params=None, headers=None, refresh=False, missing_ok=False):
        # Content-addressed cache records the original retrieval time, including on reuse.
        identity = json.dumps([url, params], sort_keys=True)
        folder = self.root / "http_cache" / sha256(identity.encode())
        meta_path = folder / "metadata.json"
        if meta_path.exists() and not refresh:
            meta = json.loads(meta_path.read_text()); data = (folder / meta["sha256"]).read_bytes()
            if sha256(data) != meta["sha256"]: raise ValueError(f"Corrupt HTTP cache: {folder}")
            return data, meta["retrieved_at"]
        data = self.get(url, params, headers, missing_ok)
        if data is None: return None, str(utcnow())
        stamp = str(utcnow()); digest = sha256(data)
        atomic_bytes(folder / digest, data)
        write_json(meta_path, {"url": url, "params": params, "retrieved_at": stamp, "sha256": digest})
        return data, stamp

## 3 · Historical universe and archive selection

Catalogue both monthly and daily namespaces, paginate every result page, and retain exact catalogue responses. Unresolved pair labels are preserved as raw exchange symbols. Current metadata is optional because it cannot establish the historical universe.

In [3]:
def list_bucket(http, prefix, delimiter=None):
    """S3 list v2 pagination; uses the bucket published by Binance's archive website."""
    files, prefixes, token = [], [], None
    while True:
        params = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
        if delimiter: params["delimiter"] = delimiter
        if token: params["continuation-token"] = token
        raw = http.get(BUCKET, params)
        # Persist exact catalogue response for historical discovery reproducibility.
        atomic_bytes(http.root / "catalog_snapshots" / (sha256(raw) + ".xml"), raw)
        tree = ET.fromstring(raw); ns = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}
        for x in tree.findall("s:Contents", ns):
            files.append({"key": x.findtext("s:Key", namespaces=ns),
                          "size": int(x.findtext("s:Size", namespaces=ns)),
                          "modified": x.findtext("s:LastModified", namespaces=ns)})
        prefixes += [x.text for x in tree.findall("s:CommonPrefixes/s:Prefix", ns)]
        if tree.findtext("s:IsTruncated", namespaces=ns) != "true": break
        nxt = tree.findtext("s:NextContinuationToken", namespaces=ns)
        if not nxt or nxt == token: raise ValueError("Invalid bucket pagination")
        token = nxt
    return files, prefixes

def infer_pair(symbol, current, overrides):
    if symbol in overrides:
        base, quote = overrides[symbol]
        return base, quote, "manual"
    if symbol in current:
        x = current[symbol]
        return x["baseAsset"], x["quoteAsset"], "current_exchange_metadata"
    # This resolves trading-pair notation, NOT a token's historical economic identity.
    for quote in KNOWN_QUOTES:
        if symbol.endswith(quote) and len(symbol) > len(quote):
            return symbol[:-len(quote)], quote, "inferred_suffix_review_required"
    return "", "", "unresolved"

def discover(http, cfg, run_dir):
    symbols = set()
    for frequency in ("monthly", "daily"):
        _, prefixes = list_bucket(http, f"data/spot/{frequency}/klines/", "/")
        symbols.update(p.rstrip("/").split("/")[-1] for p in prefixes)
    current = {}
    if cfg["FETCH_CURRENT_METADATA"]:
        raw, _ = http.cached("https://api.binance.com/api/v3/exchangeInfo", refresh=True)
        atomic_bytes(run_dir / "exchange_info.json", raw)
        current = {x["symbol"]: x for x in json.loads(raw)["symbols"]}
    rows = []
    for symbol in sorted(symbols):
        if not symbol or not all(ch.isalnum() or ch == "_" for ch in symbol):
            raise ValueError("Unsafe archive symbol")
        base, quote, method = infer_pair(symbol, current, cfg["PAIR_OVERRIDES"])
        selected = cfg["QUOTES"] is None or quote in cfg["QUOTES"]
        if cfg["SYMBOLS"] is not None: selected &= symbol in cfg["SYMBOLS"]
        rows.append({"symbol": symbol, "ticker": f"{base}-{quote}" if base else symbol,
            "base_asset": base, "quote_asset": quote, "pair_mapping": method,
            "instrument_id": "binance:spot:" + symbol,
            "current_status": current.get(symbol, {}).get("status", "unknown_or_not_current"),
            "selected": bool(selected)})
    result = pd.DataFrame(rows)
    if result.empty: raise RuntimeError("Empty universe discovery; do not continue")
    write_csv(run_dir / "instruments.csv", result)
    return result

def plan_symbol(http, symbol, cfg):
    start, end = pd.Timestamp(cfg["START"], tz="UTC"), pd.Timestamp(cfg["END_EXCLUSIVE"], tz="UTC")
    chosen, months = [], set()
    for frequency in ("monthly", "daily"):
        base_prefix = f"data/spot/{frequency}/klines/{symbol}/{cfg['INTERVAL']}/"
        if frequency == "monthly":
            files, _ = list_bucket(http, base_prefix)
        else:
            requested = set(pd.date_range(start.normalize().replace(day=1),
                (end - pd.Timedelta(nanoseconds=1)).normalize().replace(day=1), freq="MS").strftime("%Y-%m"))
            missing_months = sorted(requested - months)
            if not missing_months: continue
            prefixes = ([base_prefix + symbol + "-" + cfg["INTERVAL"] + "-" + month + "-" for month in missing_months]
                        if len(missing_months) <= 3 else [base_prefix])
            files = []
            for prefix in prefixes:
                found, _ = list_bucket(http, prefix); files.extend(found)
        for item in files:
            name = item["key"].rsplit("/", 1)[-1]
            match = re.fullmatch(re.escape(symbol + "-" + cfg["INTERVAL"] + "-") + r"(\d{4}-\d{2}(?:-\d{2})?)\.zip", name)
            if not match: continue
            date = pd.Timestamp(match[1], tz="UTC")
            stop = date + (pd.offsets.MonthBegin(1) if frequency == "monthly" else pd.Timedelta(days=1))
            if stop <= start or date >= end: continue
            month = date.strftime("%Y-%m")
            if frequency == "daily" and month in months: continue
            chosen.append({**item, "symbol": symbol, "frequency": frequency})
            if frequency == "monthly": months.add(month)
    return sorted(chosen, key=lambda x: x["key"])

def verified_archive(http, item, recheck=False):
    key = item["key"]; folder = http.root / "archives" / key
    receipt = folder / "receipt.json"
    if receipt.exists() and not recheck:
        metadata = json.loads(receipt.read_text())
        data = (folder / metadata["sha256"]).read_bytes()
        if sha256(data) != metadata["sha256"]: raise ValueError("Archive cache checksum mismatch: " + key)
        return data, metadata
    check = http.get(ARCHIVE + key + ".CHECKSUM")
    expected = check.decode().strip().split()[0].lower()
    if not re.fullmatch(r"[0-9a-f]{64}", expected): raise ValueError("Malformed publisher checksum")
    data = http.get(ARCHIVE + key)
    if sha256(data) != expected: raise ValueError("Publisher checksum mismatch: " + key)
    metadata = {"key": key, "sha256": expected, "retrieved_at": str(utcnow()),
                "publisher_checksum": check.decode().strip(), "archive_modified": item.get("modified")}
    atomic_bytes(folder / expected, data); write_json(receipt, metadata)
    return data, metadata

## 4 · Candle normalization and quality gates

Parse both timestamp precisions, verify UTC boundaries, OHLC inequalities, finite positive prices, nonnegative volume, integer trade counts and taker-volume bounds. Duplicate bars fail validation. Large valid returns are not winsorized. Internal gaps and zero-turnover days are separate diagnostics.

In [4]:
def epoch(series):
    values = pd.to_numeric(series, errors="raise").astype("int64")
    # Spot archive precision changed from milliseconds to microseconds in 2025.
    us = values.abs() >= 100_000_000_000_000
    out = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns, UTC]")
    out.loc[us] = pd.to_datetime(values[us], unit="us", utc=True)
    out.loc[~us] = pd.to_datetime(values[~us], unit="ms", utc=True)
    return out

def parse_archive(data, instrument, meta, cfg):
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        members = [x for x in archive.infolist() if x.filename.endswith(".csv")]
        if len(members) != 1 or members[0].file_size > 512_000_000:
            raise ValueError("Expected one bounded CSV in ZIP")
        with archive.open(members[0]) as f: d = pd.read_csv(f, header=None, dtype=str)
    if d.shape[1] != 12 or d.empty: raise ValueError("Unexpected Binance kline schema")
    if str(d.iloc[0, 0]).lower().replace("_", " ") == "open time": d = d.iloc[1:].copy()
    d.columns = COLS
    d["open_time"] = epoch(d.open_time); d["close_time"] = epoch(d.close_time)
    numeric = COLS[1:6] + COLS[7:11]
    for col in numeric: d[col] = pd.to_numeric(d[col], errors="raise")
    if not np.isfinite(d[numeric].to_numpy(dtype=float)).all(): raise ValueError("Nonfinite candle values")
    bad = ((d[["open", "high", "low", "close"]] <= 0).any(axis=1)
        | (d.high < d[["open", "close", "low"]].max(axis=1))
        | (d.low > d[["open", "close", "high"]].min(axis=1))
        | (d[["volume", "quote_volume", "trades", "taker_buy_volume", "taker_buy_quote_volume"]] < 0).any(axis=1)
        | (d.trades % 1 != 0)
        | (d.taker_buy_volume > d.volume * (1 + 1e-8) + 1e-8)
        | (d.taker_buy_quote_volume > d.quote_volume * (1 + 1e-8) + 1e-8))
    if bad.any(): raise ValueError(f"{int(bad.sum())} invalid OHLCV rows; archive rejected")
    delta = {"1d": pd.Timedelta(days=1), "1h": pd.Timedelta(hours=1)}[cfg["INTERVAL"]]
    if ((d.close_time < d.open_time) | (d.close_time >= d.open_time + delta)).any():
        raise ValueError("Candle close timestamp outside interval")
    if (d.open_time.astype("int64") % delta.value != 0).any(): raise ValueError("Off-grid candle timestamp")
    start, end = pd.Timestamp(cfg["START"], tz="UTC"), pd.Timestamp(cfg["END_EXCLUSIVE"], tz="UTC")
    d = d.loc[(d.open_time >= start) & (d.open_time + delta <= end)].copy()
    d["bar_end"] = d.open_time + delta
    d["date"] = d.open_time.dt.strftime("%Y-%m-%d")
    d["trades"] = d.trades.astype("int64")
    for field in ("symbol", "ticker", "base_asset", "quote_asset", "instrument_id"):
        d[field] = instrument[field]
    d["interval"] = cfg["INTERVAL"]; d["source"] = "binance_spot_archive"
    d["archive_key"] = meta["key"]; d["archive_sha256"] = meta["sha256"]
    d["retrieved_at"] = meta["retrieved_at"]
    return d.drop(columns="ignore")

def consolidate(frames):
    if not frames: return pd.DataFrame()
    d = pd.concat(frames, ignore_index=True).sort_values("open_time")
    if d.duplicated(["instrument_id", "open_time"]).any():
        raise ValueError("Overlapping candles: investigate; never silently keep first/last")
    return d

def quality(d, cfg):
    if d.empty: return {"rows": 0}
    delta = pd.Timedelta(days=1) if cfg["INTERVAL"] == "1d" else pd.Timedelta(hours=1)
    expected = pd.date_range(d.open_time.min(), d.open_time.max(), freq=delta)
    missing = expected.difference(pd.DatetimeIndex(d.open_time))
    return {"symbol": d.symbol.iloc[0], "rows": len(d), "first_bar": str(d.open_time.min()),
        "last_bar": str(d.open_time.max()), "internal_missing_bars": len(missing),
        "internal_coverage": len(d) / len(expected), "zero_volume_bars": int((d.volume == 0).sum()),
        "flat_bars": int((d.high == d.low).sum()),
        "missing_bar_examples": ";".join(map(str, missing[:10]))}

## 5 · Historical market cap and identity control

The free adapter reads Coin Metrics' actual `CapMrktCurUSD`, with source price and supply retained when present. Coverage is limited and its archive license is **CC BY-NC 4.0**: do not assume permission for commercial/institutional trading use. Review licensing for your intended deployment.

The alternative CoinGecko Pro adapter downloads daily historical caps, reference prices and global rolling-24-hour volume. Set `COINGECKO_PRO_API_KEY` in your environment—not in this notebook. Full 2021 history needs an appropriately entitled plan; Demo's 365-day window is insufficient, and current Basic access is also insufficient. There is no promise of free, complete circulating-cap history for every Binance asset.

`asset_mapping.csv` is editable and survives reruns. One row per ticker identity/validity window: `ticker`, `asset_id`, `coingecko_id`, `coinmetrics_id`, `reviewed`, `valid_from`, `valid_to_exclusive`, `note`. Only reviewed rows are used. Ticker matches alone cannot disambiguate symbols such as renamed or migrated tokens. Optional candidate lists reduce clerical work but do not approve a match.

Free-source absences and unreviewed assets remain missing. No market-cap interpolation or estimate based on current circulating supply is used.

In [5]:
def mapping_template(instruments, path):
    """Seed unambiguous identities only. Every other asset requires human review."""
    seeds = {"BTC": ("bitcoin", "btc"), "ETH": ("ethereum", "eth"),
        "LTC": ("litecoin", "ltc"), "DOGE": ("dogecoin", "doge"),
        "ADA": ("cardano", "ada"), "XRP": ("ripple", "xrp"),
        "BCH": ("bitcoin-cash", "bch"), "LINK": ("chainlink", "link"),
        "SOL": ("solana", "sol"), "BNB": ("binancecoin", "bnb")}
    rows = []
    for base in sorted(set(instruments.base_asset) - {""}):
        cg, cm = seeds.get(base, ("", ""))
        rows.append({"ticker": base, "asset_id": "coingecko:" + cg if cg else "",
            "coingecko_id": cg, "coinmetrics_id": cm, "reviewed": bool(cg),
            "valid_from": "2021-01-01", "valid_to_exclusive": "2100-01-01",
            "note": "seed; verify for your research" if cg else "REVIEW identity, migrations, contract and valid dates"})
    if not Path(path).exists(): write_csv(path, pd.DataFrame(rows))
    else:
        existing = pd.read_csv(path, keep_default_na=False)
        new = pd.DataFrame(rows)
        new = new.loc[~new.ticker.isin(existing.ticker)]
        if len(new): write_csv(path, pd.concat([existing, new], ignore_index=True))
    return pd.read_csv(path, keep_default_na=False)

def suggest_market_cap_ids(http, mapping, run_dir):
    """Candidates are suggestions only: ticker matches must not approve identities."""
    raw, _ = http.cached("https://api.coingecko.com/api/v3/coins/list")
    listing = pd.DataFrame(json.loads(raw))
    listing["symbol"] = listing.symbol.str.upper()
    suggestions = mapping[["ticker", "reviewed"]].merge(
        listing.rename(columns={"symbol": "ticker", "id": "candidate_coingecko_id", "name": "candidate_name"}),
        on="ticker", how="left")
    suggestions["warning"] = "NOT VERIFIED: check contract, migration and historical validity before copying to asset_mapping.csv"
    write_csv(run_dir / "market_cap_mapping_candidates.csv", suggestions)

def cap_coverage_report(run_dir, caps):
    registry = pd.read_csv(Path(run_dir) / "partition_manifest.csv")
    available = {}
    if len(caps):
        c = caps.copy(); c["timestamp"] = pd.to_datetime(c.timestamp, utc=True)
        c = c.loc[c.timestamp.eq(c.timestamp.dt.normalize())]
        for ticker, g in c.groupby("ticker"): available[ticker] = set(g.timestamp.dt.strftime("%Y-%m-%d"))
    rows = []
    for item in registry.itertuples():
        if item.status != "ok": continue
        d = pd.read_csv(Path(run_dir) / item.path, usecols=["base_asset", "date"])
        base = d.base_asset.iloc[0]; dates = set(d.date)
        matched = len(dates & available.get(base, set()))
        rows.append({"symbol": item.symbol, "base_asset": base, "observed_days": len(dates),
            "days_with_exact_cap": matched, "cap_coverage_fraction": matched / len(dates)})
    result = pd.DataFrame(rows)
    write_csv(Path(run_dir) / "market_cap_coverage.csv", result)
    return result

def validate_mapping(mapping):
    required = {"ticker", "asset_id", "coingecko_id", "coinmetrics_id", "reviewed", "valid_from", "valid_to_exclusive"}
    if not required.issubset(mapping): raise ValueError("Mapping columns missing")
    d = mapping.loc[mapping.reviewed.astype(str).str.lower().eq("true")].copy()
    for row in d.itertuples():
        if not row.asset_id or not row.ticker or not all(ch.isalnum() or ch in "_-" for ch in row.ticker):
            raise ValueError("Invalid reviewed identity")
        if pd.Timestamp(row.valid_from) >= pd.Timestamp(row.valid_to_exclusive): raise ValueError("Empty mapping validity window")
        for identifier in (row.coingecko_id, row.coinmetrics_id):
            if identifier and not re.fullmatch(r"[a-zA-Z0-9_-]+", identifier): raise ValueError("Unsafe provider ID")
    for ticker, group in d.groupby("ticker"):
        g = group.sort_values("valid_from")
        if len(g) > 1 and any(pd.to_datetime(g.valid_from.iloc[1:]).to_numpy() < pd.to_datetime(g.valid_to_exclusive.iloc[:-1]).to_numpy()):
            raise ValueError("Overlapping identity windows for " + ticker)
    return d

def market_caps(http, mapping, cfg, run_dir):
    rows, errors = [], []
    reviewed = validate_mapping(mapping)
    write_csv(run_dir / "asset_mapping_used.csv", mapping)
    for r in reviewed.itertuples():
        start = max(pd.Timestamp(cfg["START"], tz="UTC"), pd.Timestamp(r.valid_from, tz="UTC"))
        end = min(pd.Timestamp(cfg["END_EXCLUSIVE"], tz="UTC"), pd.Timestamp(r.valid_to_exclusive, tz="UTC"))
        if start >= end: continue
        try:
            if cfg["CAP_SOURCE"] == "coinmetrics":
                if not r.coinmetrics_id: raise ValueError("No reviewed Coin Metrics ID")
                raw, retrieved = http.cached(CM + r.coinmetrics_id + ".csv", refresh=cfg["REFRESH_CAPS"], missing_ok=True)
                if raw is None: raise ValueError("Asset absent from free Coin Metrics archive")
                source = pd.read_csv(io.BytesIO(raw))
                if "CapMrktCurUSD" not in source: raise ValueError("Circulating-cap metric unavailable")
                d = pd.DataFrame({"timestamp": pd.to_datetime(source.time, utc=True),
                    "market_cap_usd": pd.to_numeric(source.CapMrktCurUSD, errors="coerce"),
                    "reference_price_usd": pd.to_numeric(source.get("PriceUSD", np.nan), errors="coerce"),
                    "circulating_supply": pd.to_numeric(source.get("SplyCur", np.nan), errors="coerce"),
                    "global_volume_24h_usd": np.nan})
                provider_id = r.coinmetrics_id
            elif cfg["CAP_SOURCE"] == "coingecko_pro":
                key = os.environ.get("COINGECKO_PRO_API_KEY")
                if not key: raise AccessBlocked("Set COINGECKO_PRO_API_KEY; historical entitlement required")
                if not r.coingecko_id: raise ValueError("No reviewed CoinGecko ID")
                raw, retrieved = http.cached(CG + f"/coins/{r.coingecko_id}/market_chart/range",
                    {"vs_currency": "usd", "from": int(start.timestamp()), "to": int(end.timestamp()), "interval": "daily"},
                    {"x-cg-pro-api-key": key}, refresh=cfg["REFRESH_CAPS"])
                source = json.loads(raw)
                def series(field, name):
                    x = pd.DataFrame(source[field], columns=["timestamp", name])
                    x["timestamp"] = pd.to_datetime(x.timestamp, unit="ms", utc=True)
                    return x.set_index("timestamp")[name]
                d = pd.concat([series("market_caps", "market_cap_usd"), series("prices", "reference_price_usd"),
                    series("total_volumes", "global_volume_24h_usd")], axis=1, sort=True).reset_index()
                d["circulating_supply"] = np.nan  # never infer from today's supply
                provider_id = r.coingecko_id
            else: raise ValueError("CAP_SOURCE must be coinmetrics or coingecko_pro")
            d = d.loc[(d.timestamp >= start) & (d.timestamp < end)].copy()
            d["market_cap_usd"] = pd.to_numeric(d.market_cap_usd, errors="raise")
            invalid = ~np.isfinite(d.market_cap_usd) | (d.market_cap_usd <= 0)
            if invalid.any(): errors.append({"ticker": r.ticker, "issue": f"{invalid.sum()} missing/nonpositive cap observations excluded"})
            d = d.loc[~invalid].copy()
            if d.empty: raise ValueError("No positive historical market caps in requested period")
            if d.timestamp.duplicated().any(): raise ValueError("Duplicate provider timestamps")
            d["asset_id"] = r.asset_id; d["ticker"] = r.ticker
            d["provider"] = cfg["CAP_SOURCE"]; d["provider_id"] = provider_id; d["retrieved_at"] = retrieved
            rows.append(d[CAP_COLUMNS])
        except AccessBlocked: raise
        except Exception as exc: errors.append({"ticker": r.ticker, "issue": str(exc)})
    caps = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=CAP_COLUMNS)
    write_csv(run_dir / "market_cap_observations.csv.gz", caps)
    write_csv(run_dir / "market_cap_observations.csv", caps)
    if isinstance(cfg, dict) and "ROOT" in cfg:
        root_dir = Path(cfg["ROOT"]).resolve()
        write_csv(root_dir / "market_cap_observations.csv.gz", caps)
        write_csv(root_dir / "market_cap_observations.csv", caps)
    write_csv(run_dir / "market_cap_issues.csv", pd.DataFrame(errors, columns=["ticker", "issue"]))
    return caps, errors

## 6 · A portable data interface

The canonical output is a long table partitioned by exchange symbol, not thousands of ad hoc wide spreadsheets. `load_panel` pivots one field into usual pair-ticker columns and restores the calendar so missing days remain NaN. `open_time` labels the beginning of a candle; OHLCV is not complete until `bar_end`.

Caps remain a separate timestamped table. The optional comparison join is **descriptive**, exact-date, midnight-only and never an availability-aware trading signal. Coin Metrics and CoinGecko can have different end-of-day conventions; reconcile them and model publication lags before strategy use.

In [6]:
def load_long(run_dir, quote=None):
    root = Path(run_dir)
    registry = pd.read_csv(root / "partition_manifest.csv")
    frames = []
    for r in registry.itertuples():
        if r.status != "ok": continue
        if quote is not None and r.quote_asset != quote: continue
        path = root / r.path
        if sha256(path.read_bytes()) != r.sha256: raise ValueError("Output partition modified: " + str(path))
        frames.append(pd.read_csv(path, parse_dates=["open_time", "close_time", "bar_end", "retrieved_at"]))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def load_panel(run_dir, field="close", quote="USDT"):
    d = load_long(run_dir, quote)
    if d.empty: return pd.DataFrame()
    if field not in {"open", "high", "low", "close", "volume", "quote_volume", "trades", "taker_buy_volume", "taker_buy_quote_volume"}:
        raise ValueError("Unsupported OHLCV field")
    panel = d.pivot(index="open_time", columns="ticker", values=field).sort_index()
    delta = "1D" if d.interval.iloc[0] == "1d" else "1h"
    return panel.reindex(pd.date_range(panel.index.min(), panel.index.max(), freq=delta))

def exact_cap_comparison(daily, caps):
    """Descriptive date match only: NOT a point-in-time availability assertion."""
    if caps.empty: return pd.DataFrame()
    c = caps.copy(); c["timestamp"] = pd.to_datetime(c.timestamp, utc=True)
    # Keep only actual midnight samples. Never relabel intraday snapshots as EOD.
    c = c.loc[c.timestamp.eq(c.timestamp.dt.normalize())].copy()
    c["date"] = c.timestamp.dt.strftime("%Y-%m-%d")
    if c.duplicated(["ticker", "date"]).any(): raise ValueError("Ambiguous cap identity on date")
    return daily.merge(c.rename(columns={"ticker": "base_asset"}), on=["base_asset", "date"], how="left", validate="many_to_one")

def daily_research_table(run_dir, quote):
    """Aggregate one partition at a time; never load the whole hourly panel to plot."""
    root = Path(run_dir); registry = pd.read_csv(root / "partition_manifest.csv")
    frames = []
    fields = ["ticker", "base_asset", "open_time", "interval", "close", "high", "low",
              "quote_volume", "volume", "trades", "taker_buy_quote_volume"]
    for row in registry.itertuples():
        if row.status != "ok" or row.quote_asset != quote: continue
        path = root / row.path
        if sha256(path.read_bytes()) != row.sha256: raise ValueError("Output partition modified: " + str(path))
        d = pd.read_csv(path, usecols=fields)
        d["day"] = pd.to_datetime(d.open_time, utc=True).dt.floor("D")
        g = d.groupby(["ticker", "base_asset", "day"])
        x = g.agg(close=("close", "last"), high=("high", "max"), low=("low", "min"),
            quote_volume=("quote_volume", "sum"), volume=("volume", "sum"), trades=("trades", "sum"),
            taker_buy_quote_volume=("taker_buy_quote_volume", "sum"), bars=("close", "count")).reset_index()
        required = 24 if d.interval.iloc[0] == "1h" else 1
        frames.append(x.loc[x.bars.eq(required)])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 7 · Twelve research views: what to look for

| Figure | Teaching question |
|---|---|
| Universe breadth | Is expanding coverage being mistaken for a stable opportunity set? |
| Coverage heatmap | Which histories are long, recent or incomplete? |
| First/last observation cohorts | Does the sample systematically omit disappeared assets? |
| Missingness versus zero volume | Is a quiet market being confused with a failed download? |
| Yearly liquidity distributions | How much of the sample lives in the thinly traded tail? |
| Top-five share and HHI | Does headline breadth conceal concentrated activity? |
| Liquidity versus volatility | Do different activity levels coincide with different variability? |
| Taker-flow density | Are low-turnover observations dominated by one-sided executions? |
| Return correlations | Are apparently distinct assets moving together? |
| Market-cap coverage | Does the size sample represent only a subset of downloaded coins? |
| Size versus venue turnover | Are market size and Binance-specific liquidity being conflated? |
| Range and average trade size | What do candle ranges and execution sizes reveal—and not reveal? |

Each plot is exported as a PNG with a written interpretation. No synthetic data enter live outputs. Coverage-selected subsets are labeled, and all diagnostics are descriptive full-sample summaries, not out-of-sample evidence.

In [7]:
def research_plots(run_dir, quote="USDT", show=True):
    """Figures are descriptive, not portfolio performance. One quote; no cross-quote sums."""
    root = Path(run_dir); out = root / "figures"; out.mkdir(exist_ok=True)
    x = daily_research_table(root, quote)
    if x.empty:
        print("No candles for research quote", quote); return []
    plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
        "axes.facecolor": "#fcfcfd", "axes.labelcolor": "#374151", "axes.grid": True,
        "grid.alpha": .18, "font.size": 10, "figure.facecolor": "white"})
    dates = pd.date_range(x.day.min(), x.day.max(), freq="D")
    close = x.pivot(index="day", columns="ticker", values="close").reindex(dates)
    vol = x.pivot(index="day", columns="ticker", values="quote_volume").reindex(dates)
    valid = close.notna(); ret = close.pct_change(fill_method=None)
    count = valid.sum(axis=1)
    captions, figures = [], []
    def hist(ax, values, bins=40, **kwargs):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if len(values) and np.ptp(values) < 1e-10 * max(1, np.abs(values).max()):
            center = float(values.mean()); width = max(abs(center) * .01, .01)
            kwargs["range"] = (center - width, center + width)
        ax.hist(values, bins=bins, **kwargs)
    def save(fig, name, title, note):
        fig.suptitle(title, fontsize=15, fontweight="bold", x=.08, ha="left")
        fig.text(.08, .015, note, fontsize=9, color="#4b5563", wrap=True)
        fig.tight_layout(rect=(0, .09, 1, .94))
        fig.savefig(out / (name + ".png"), dpi=160)
        if show:
            try:
                from IPython.display import display
                display(fig)
            except Exception:
                plt.show()
        plt.close(fig); figures.append(name); captions.append({"figure": name, "interpretation": note})
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(count.index, count, color="#2563eb", alpha=.2); ax.plot(count, color="#2563eb")
    ax.set_ylabel("Pairs with a complete observed day")
    save(fig, "01_universe_breadth", f"Breadth changes through time · {quote}",
         "Observed coverage is not the historical investable universe. Dips can reflect missing downloads or exchange outages.")
    fig, ax = plt.subplots(figsize=(12, 7))
    chosen = valid.sum().sort_values(ascending=False).head(50).index
    heat = valid[chosen].resample("MS").mean().T
    im = ax.imshow(heat, aspect="auto", vmin=0, vmax=1, cmap="Blues")
    ax.set_yticks(range(len(chosen)), chosen, fontsize=7)
    ticks = np.arange(0, len(heat.columns), 6)
    ax.set_xticks(ticks, [heat.columns[i].strftime("%Y-%m") for i in ticks], rotation=45)
    fig.colorbar(im, ax=ax, label="Fraction of calendar days observed")
    save(fig, "02_coverage_heatmap", "Where are the holes?", "50 longest observed histories; selection is descriptive and favors long-lived pairs. Missing bars are never filled.")
    spans = x.groupby("ticker").day.agg(["min", "max", "count"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    spans["min"].dt.year.value_counts().sort_index().plot.bar(ax=axes[0], color="#0d9488")
    spans["max"].dt.year.value_counts().sort_index().plot.bar(ax=axes[1], color="#f59e0b")
    axes[0].set_title("First observed year"); axes[1].set_title("Last observed year")
    save(fig, "03_observation_cohorts", "Entry and disappearance of observed histories", "First/last bars are sample boundaries, not verified listing/delisting dates. 2021 is left-censored.")
    span_days = (spans["max"] - spans["min"]).dt.days + 1
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    hist(axes[0], 100 * (1 - spans["count"] / span_days), bins=30, color="#ef4444")
    axes[0].set_xlabel("Missing internal calendar days (%)")
    zero = x.groupby("ticker").quote_volume.apply(lambda z: 100 * z.eq(0).mean())
    hist(axes[1], zero, bins=30, color="#8b5cf6"); axes[1].set_xlabel("Zero-turnover observed days (%)")
    save(fig, "04_quality_distributions", "No trade is different from no observation", "Internal gaps exclude dates before first and after last observation. Zero volume remains a real recorded value.")
    positive = x.loc[x.quote_volume.gt(0)].copy(); positive["year"] = positive.day.dt.year
    years = sorted(positive.year.unique())
    fig, ax = plt.subplots(figsize=(12, 5))
    if years: ax.boxplot([np.log10(positive.loc[positive.year.eq(y), "quote_volume"]) for y in years], tick_labels=years, showfliers=False)
    ax.set_ylabel(f"log10 daily turnover ({quote})")
    save(fig, "05_liquidity_distribution", "The liquidity distribution—not just its mean", "All positive pair-days, so longer histories receive more weight. Whiskers are 1.5×IQR; outliers are hidden, not deleted.")
    daily_total = vol.sum(axis=1, min_count=1).replace(0, np.nan)
    shares = vol.div(daily_total, axis=0)
    top5 = shares.apply(lambda z: z.nlargest(5).sum(min_count=1), axis=1)
    hhi = shares.pow(2).sum(axis=1, min_count=1)
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(top5.index, 100 * top5, color="#f59e0b"); axes[0].set_ylabel("Top 5 turnover share (%)")
    axes[1].plot(hhi, color="#7c3aed"); axes[1].set_ylabel("Turnover concentration (HHI)")
    save(fig, "06_volume_concentration", "Many listings need not mean broad liquidity", "Ranks are recomputed each day among observed pairs; missing coverage can artificially raise concentration.")
    median_vol = vol.median(); volatility = ret.std() * np.sqrt(365) * 100
    eligible = ret.count().ge(90) & median_vol.gt(0) & volatility.notna()
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(median_vol[eligible], volatility[eligible], alpha=.45, s=24, color="#0d9488")
    ax.set_xscale("log"); ax.set_xlabel(f"Median daily turnover ({quote})"); ax.set_ylabel("Annualized daily-return volatility (%)")
    save(fig, "07_liquidity_volatility", "How liquidity and price variability coexist", "At least 90 valid one-day returns per pair; full-sample diagnostic, not a predictive or causal relationship.")
    fig, ax = plt.subplots(figsize=(10, 6))
    y = x.loc[x.quote_volume.gt(0)].copy()
    imbalance = 2 * y.taker_buy_quote_volume / y.quote_volume - 1
    if len(y):
        im = ax.hexbin(np.log10(y.quote_volume), imbalance, gridsize=55, bins="log", mincnt=1, cmap="viridis")
        fig.colorbar(im, ax=ax, label="Pair-day count (log color)")
    ax.set_xlabel(f"log10 daily turnover ({quote})"); ax.set_ylabel("2 × taker-buy share − 1")
    save(fig, "08_taker_flow", "Aggressive buying versus selling", "Taker imbalance measures which side initiated executed trades; it is not an order-book spread or net capital flow.")
    chosen = ret.count().loc[lambda z: z >= 90].sort_values(ascending=False).head(20).index
    fig, ax = plt.subplots(figsize=(10, 8))
    if len(chosen) >= 2:
        corr = ret[chosen].corr(min_periods=90)
        im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
        ax.set_xticks(range(len(chosen)), chosen, rotation=90, fontsize=8); ax.set_yticks(range(len(chosen)), chosen, fontsize=8)
        fig.colorbar(im, ax=ax, label="Pearson daily-return correlation")
    else: ax.text(.1, .5, "Insufficient overlapping histories", transform=ax.transAxes)
    save(fig, "09_correlation", "Similar prices? Compare returns, not price levels", "20 longest usable return histories; pairwise minimum 90 observations. Differing overlap can make this matrix non-PSD.")
    cap_file = root / "market_cap_observations.csv"
    if not cap_file.exists():
        cap_file = root / "market_cap_observations.csv.gz"
    caps = pd.read_csv(cap_file) if cap_file.exists() else pd.DataFrame(columns=CAP_COLUMNS)
    x["date"] = x.day.dt.strftime("%Y-%m-%d")
    joined = exact_cap_comparison(x, caps)
    fig, ax = plt.subplots(figsize=(12, 5))
    if not joined.empty:
        fraction = joined.groupby("day").market_cap_usd.apply(lambda z: z.notna().mean())
        ax.plot(fraction.index, 100 * fraction, color="#2563eb"); ax.set_ylim(0, 105)
    else: ax.text(.1, .5, "No matched historical market caps—do not infer missing values", transform=ax.transAxes)
    ax.set_ylabel("Observed pair-days with same-date cap (%)")
    save(fig, "10_cap_coverage", "Market-cap coverage is part of the result", "Only reviewed identities and exact UTC-midnight cap dates match. No forward/back fill; midnight cap precedes that day's candle close.")
    matched = joined.dropna(subset=["market_cap_usd"]) if not joined.empty else pd.DataFrame()
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    if not matched.empty:
        cross = matched.loc[matched.day.eq(matched.day.max()) & matched.quote_volume.gt(0)]
        axes[0].scatter(cross.market_cap_usd, cross.quote_volume, color="#7c3aed", alpha=.6)
        axes[0].set_xscale("log"); axes[0].set_yscale("log")
        hist(axes[1], np.log10(matched.market_cap_usd), bins=40, color="#2563eb")
        axes[0].set_title("Latest matched day: " + str(matched.day.max().date()))
    else:
        for ax in axes: ax.text(.1, .5, "Market-cap observations unavailable", transform=ax.transAxes)
    axes[0].set_xlabel("Circulating market cap (USD)"); axes[0].set_ylabel(f"Binance daily turnover ({quote})")
    axes[1].set_xlabel("log10 market cap (USD), matched pair-days")
    save(fig, "11_size_liquidity", "Size and venue liquidity are different dimensions", f"Cap describes the whole asset; turnover describes Binance/{quote} only. {quote} is not silently treated as USD.")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    intraday_range = 100 * (x.high - x.low) / x.close
    hist(axes[0], np.log10(1 + intraday_range), bins=50, color="#ec4899"); axes[0].set_xlabel("log10(1 + daily high–low range in %)")
    valid_trades = x.loc[x.trades.gt(0) & x.quote_volume.gt(0)]
    hist(axes[1], np.log10(valid_trades.quote_volume / valid_trades.trades), bins=50, color="#0d9488")
    axes[1].set_xlabel(f"log10 average executed trade size ({quote})")
    save(fig, "12_range_trade_size", "Two additional views of market activity", "Range is not bid–ask spread; turnover/trade count is an average execution size, not a transaction-cost estimate.")
    write_csv(out / "figure_guide.csv", pd.DataFrame(captions))
    if not joined.empty:
        write_csv(root / f"daily_cap_comparison_{quote}.csv.gz", joined)
        write_csv(root / f"daily_cap_comparison_{quote}.csv", joined)
    return figures

## 8 · Orchestrate and checkpoint

Each run writes a separate directory. `archive_acquisition_finished` means the selected published files were processed; it does not mean that all desired historical bars exist. `market_caps_complete` is assessed separately against observed pair-days. Use the coverage files, not a green status alone.

In [8]:
def run_download(cfg):
    cfg = dict(cfg)
    if cfg["INTERVAL"] not in ("1d", "1h"): raise ValueError("Validated intervals: 1d and 1h")
    start, end = pd.Timestamp(cfg["START"], tz="UTC"), pd.Timestamp(cfg["END_EXCLUSIVE"], tz="UTC")
    if start >= end or end > utcnow().normalize(): raise ValueError("Request completed UTC days only")
    root = Path(cfg["ROOT"]).resolve(); root.mkdir(parents=True, exist_ok=True)
    # Single writer per output root with automated stale lock cleanup on crashed sessions
    lock = root / "writer.lock"
    if lock.exists():
        try:
            old_pid = int(lock.read_text().strip())
            os.kill(old_pid, 0)
            raise RuntimeError(f"Another process (PID {old_pid}) is currently running acquisition in {root}")
        except (ValueError, OSError):
            lock.unlink(missing_ok=True)
    try:
        fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        with os.fdopen(fd, "w") as f: f.write(str(os.getpid()))
    except FileExistsError:
        raise RuntimeError(f"Acquisition locked by another process in {root}. If no download is running, delete {lock}")
    run_dir = root / "runs" / utcnow().strftime("%Y%m%dT%H%M%S%fZ")
    run_dir.mkdir(parents=True); http = HTTP(root / "raw", cfg["REQUEST_PAUSE_SECONDS"])
    status = {"state": "running", "version": VERSION, "started_at": str(utcnow()), "run_dir": str(run_dir),
              "scope": "published Binance spot archives only; not all crypto assets", "market_caps_complete": False}
    partitions, audit, plans, checks, failures = [], [], [], [], []
    write_json(run_dir / "config.json", cfg)
    write_json(run_dir / "environment.json", {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__})
    write_json(run_dir / "run_status.json", status)
    try:
        instruments = discover(http, cfg, run_dir)
        mapping = mapping_template(instruments, root / "asset_mapping.csv")
        if cfg.get("SUGGEST_MARKET_CAP_IDS", False): suggest_market_cap_ids(http, mapping, run_dir)
        for number, row in enumerate(instruments.loc[instruments.selected].to_dict("records"), 1):
            symbol = row["symbol"]; print(f"[{number}] {symbol}", flush=True)
            try:
                items = plan_symbol(http, symbol, cfg); plans.extend(items)
                frames = []
                for item in items:
                    raw, receipt = verified_archive(http, item, cfg["RECHECK_ARCHIVES"])
                    frames.append(parse_archive(raw, row, receipt, cfg)); audit.append(receipt)
                d = consolidate(frames)
                if d.empty:
                    partitions.append({"symbol": symbol, "quote_asset": row["quote_asset"], "status": "no_published_rows", "path": "", "sha256": "", "rows": 0})
                    continue
                relative = f"ohlcv/{symbol}.csv.gz"; write_csv(run_dir / relative, d)
                write_csv(run_dir / f"ohlcv/{symbol}.csv", d)
                root_ohlcv = root / "ohlcv"; root_ohlcv.mkdir(parents=True, exist_ok=True)
                write_csv(root_ohlcv / f"{symbol}.csv.gz", d)
                write_csv(root_ohlcv / f"{symbol}.csv", d)
                partitions.append({"symbol": symbol, "quote_asset": row["quote_asset"], "status": "ok", "path": relative,
                    "sha256": sha256((run_dir / relative).read_bytes()), "rows": len(d)})
                checks.append(quality(d, cfg))
            except AccessBlocked: raise
            except Exception as exc:
                failures.append({"symbol": symbol, "issue": str(exc)})
                partitions.append({"symbol": symbol, "quote_asset": row["quote_asset"], "status": "failed", "path": "", "sha256": "", "rows": 0})
            # Checkpoint per instrument. Unfinished runs are never advertised as final datasets.
            write_csv(run_dir / "partition_manifest.csv", pd.DataFrame(partitions))
            write_csv(run_dir / "failures.csv", pd.DataFrame(failures, columns=["symbol", "issue"]))
            status.update({"last_processed_symbol": symbol, "processed_instruments": len(partitions), "updated_at": str(utcnow())})
            write_json(run_dir / "run_status.json", status)
        selected_bases = set(instruments.loc[instruments.selected, "base_asset"])
        caps, cap_errors = market_caps(http, mapping.loc[mapping.ticker.isin(selected_bases)], cfg, run_dir)
        status.update({"state": "partial" if failures else "archive_acquisition_finished",
            "selected_instruments": int(instruments.selected.sum()), "successful_partitions": sum(p["status"] == "ok" for p in partitions),
            "ohlcv_rows": sum(p["rows"] for p in partitions), "market_cap_rows": len(caps),
            "mapped_cap_tickers": caps.ticker.nunique(), "cap_issue_count": len(cap_errors),
            "note": "Acquisition finished does not establish archive completeness, historical identity or investability."})
        if status["successful_partitions"] == 0: status["state"] = "empty"
        write_csv(run_dir / "quality_report.csv", pd.DataFrame(checks))
        write_csv(run_dir / "partition_manifest.csv", pd.DataFrame(partitions))
        coverage = cap_coverage_report(run_dir, caps)
        status["market_caps_complete"] = bool(len(coverage) and coverage.cap_coverage_fraction.eq(1).all())
        if cfg["MAKE_PLOTS"]:
            status["figures"] = research_plots(run_dir, cfg["RESEARCH_QUOTE"], cfg["SHOW_PLOTS"])
            root_figures = root / "figures"; root_figures.mkdir(parents=True, exist_ok=True)
            for fpath in (run_dir / "figures").glob("*"):
                if fpath.is_file(): atomic_bytes(root_figures / fpath.name, fpath.read_bytes())
        print(json.dumps({**status, "run_dir": Path(os.path.relpath(run_dir, ".")).as_posix()}, indent=2, default=str))
        return run_dir
    except BaseException as exc:
        status["state"] = "blocked" if isinstance(exc, AccessBlocked) else "interrupted_or_failed"
        status["error"] = str(exc); raise
    finally:
        status["finished_at"] = str(utcnow())
        write_json(run_dir / "run_status.json", status)
        write_json(run_dir / "request_audit.json", http.audit)
        write_json(run_dir / "archive_receipts.json", audit)
        write_csv(run_dir / "archive_plan.csv", pd.DataFrame(plans))
        write_csv(run_dir / "partition_manifest.csv", pd.DataFrame(partitions, columns=["symbol", "quote_asset", "status", "path", "sha256", "rows"]))
        write_csv(run_dir / "quality_report.csv", pd.DataFrame(checks))
        write_csv(run_dir / "failures.csv", pd.DataFrame(failures, columns=["symbol", "issue"]))
        hashes = {str(p.relative_to(run_dir)): sha256(p.read_bytes()) for p in run_dir.rglob("*") if p.is_file() and p.name != "file_hashes.json"}
        write_json(run_dir / "file_hashes.json", hashes)
        lock.unlink(missing_ok=True)

## 9 · Offline regression tests

Sixteen tests exercise timestamps, headers, zero-volume retention, invalid candles, duplicate rejection, completed-bar boundaries, symbol notation, gap-aware panels, identity-window collisions, exact-date cap joins, checksum/cache corruption, monthly precedence and all twelve plotting paths. Test fixtures are synthetic and are confined to temporary directories; they are never exported as market observations.

In [9]:
import unittest, tempfile, io, zipfile, json
from pathlib import Path
from unittest.mock import patch
import numpy as np
import pandas as pd
import types
e = types.SimpleNamespace(**globals())

CFG = {"START": "2021-01-01", "END_EXCLUSIVE": "2026-01-01", "INTERVAL": "1d"}
INSTRUMENT = {"symbol": "BTCUSDT", "ticker": "BTC-USDT", "base_asset": "BTC", "quote_asset": "USDT", "instrument_id": "binance:spot:BTCUSDT"}
META = {"key": "fixture.zip", "sha256": "fixture", "retrieved_at": "2026-01-01T00:00:00Z"}

def zipped(row, header=False):
    b = io.BytesIO()
    with zipfile.ZipFile(b, "w") as z:
        z.writestr("fixture.csv", (",".join(e.COLS) + "\n" if header else "") + ",".join(map(str, row)) + "\n")
    return b.getvalue()

def candle(date="2021-01-01", micro=False):
    stamp = pd.Timestamp(date, tz="UTC").value // (1000 if micro else 1_000_000)
    length = 86_400 * (1_000_000 if micro else 1000)
    return [stamp, 100, 110, 90, 105, 10, stamp + length - 1, 1000, 20, 5, 500, 0]

class Tests(unittest.TestCase):
    def test_milliseconds(self):
        d = e.parse_archive(zipped(candle()), INSTRUMENT, META, CFG)
        self.assertEqual(d.open_time.iloc[0], pd.Timestamp("2021-01-01", tz="UTC"))
        self.assertEqual(d.bar_end.iloc[0], pd.Timestamp("2021-01-02", tz="UTC"))
    def test_microseconds_and_header(self):
        d = e.parse_archive(zipped(candle("2025-01-01", True), True), INSTRUMENT, META, CFG)
        self.assertEqual(d.open_time.iloc[0].year, 2025)
        self.assertEqual(d.close_time.iloc[0].microsecond, 999999)
    def test_invalid_ohlc(self):
        row = candle(); row[2] = 95
        with self.assertRaises(ValueError): e.parse_archive(zipped(row), INSTRUMENT, META, CFG)
    def test_negative_volume(self):
        row = candle(); row[5] = -1
        with self.assertRaises(ValueError): e.parse_archive(zipped(row), INSTRUMENT, META, CFG)
    def test_zero_volume_kept(self):
        row = candle()
        for i in (5, 7, 8, 9, 10): row[i] = 0
        d = e.parse_archive(zipped(row), INSTRUMENT, META, CFG)
        self.assertEqual(len(d), 1)
    def test_duplicate_rejected(self):
        d = e.parse_archive(zipped(candle()), INSTRUMENT, META, CFG)
        with self.assertRaises(ValueError): e.consolidate([d, d])
    def test_incomplete_day_excluded(self):
        cfg = dict(CFG, END_EXCLUSIVE="2021-01-01")
        self.assertTrue(e.parse_archive(zipped(candle()), INSTRUMENT, META, cfg).empty)
    def test_off_grid_rejected(self):
        row = candle(); row[0] += 1; row[6] += 1
        with self.assertRaises(ValueError): e.parse_archive(zipped(row), INSTRUMENT, META, CFG)
    def test_pair_labels(self):
        self.assertEqual(e.infer_pair("BTCUSDT", {}, {})[:2], ("BTC", "USDT"))
        self.assertEqual(e.infer_pair("BTCSOMETHING", {}, {})[2], "unresolved")
    def test_calendar_gap_not_filled(self):
        d1 = e.parse_archive(zipped(candle()), INSTRUMENT, META, CFG)
        d3 = e.parse_archive(zipped(candle("2021-01-03")), INSTRUMENT, META, CFG)
        d = e.consolidate([d1, d3])
        self.assertEqual(e.quality(d, CFG)["internal_missing_bars"], 1)
        with tempfile.TemporaryDirectory() as tmp:
            root = Path(tmp); e.write_csv(root / "part.csv.gz", d)
            e.write_csv(root / "partition_manifest.csv", pd.DataFrame([{"status": "ok", "quote_asset": "USDT", "path": "part.csv.gz", "sha256": e.sha256((root / "part.csv.gz").read_bytes())}]))
            panel = e.load_panel(root)
            self.assertTrue(panel.iloc[1].isna().all())
            self.assertTrue(panel.pct_change(fill_method=None).iloc[2].isna().all())
    def test_conflicting_identity_windows(self):
        row = {"ticker": "BTC", "asset_id": "bitcoin", "coingecko_id": "bitcoin", "coinmetrics_id": "btc", "reviewed": True, "valid_from": "2021-01-01", "valid_to_exclusive": "2030-01-01"}
        with self.assertRaises(ValueError): e.validate_mapping(pd.DataFrame([row, row]))
    def test_caps_no_forward_fill(self):
        daily = pd.DataFrame({"base_asset": ["BTC", "BTC"], "date": ["2021-01-01", "2021-01-02"]})
        caps = pd.DataFrame({"ticker": ["BTC"], "timestamp": ["2021-01-01T00:00:00Z"], "market_cap_usd": [100.]})
        joined = e.exact_cap_comparison(daily, caps)
        self.assertTrue(pd.isna(joined.market_cap_usd.iloc[1]))
    def test_checksum_rejects_corruption(self):
        with tempfile.TemporaryDirectory() as tmp:
            h = e.HTTP(tmp)
            with patch.object(h, "get", side_effect=[b"0" * 64 + b" file.zip", b"wrong"]):
                with self.assertRaises(ValueError): e.verified_archive(h, {"key": "test.zip"})
    def test_cache_detects_local_corruption(self):
        with tempfile.TemporaryDirectory() as tmp:
            h = e.HTTP(tmp)
            with patch.object(h, "get", return_value=b"original"):
                h.cached("https://example.test/data")
            path = next((Path(tmp) / "http_cache").glob("*/" + e.sha256(b"original")))
            e.atomic_bytes(path, b"corrupt")
            with self.assertRaises(ValueError): h.cached("https://example.test/data")
    def test_monthly_wins_daily(self):
        def listing(http, prefix):
            if "monthly" in prefix: return [{"key": prefix + "BTCUSDT-1d-2021-01.zip", "size": 1}], []
            folder = "data/spot/daily/klines/BTCUSDT/1d/"
            return [{"key": folder + "BTCUSDT-1d-2021-01-01.zip", "size": 1}, {"key": folder + "BTCUSDT-1d-2021-02-01.zip", "size": 1}], []
        with patch.dict(e.plan_symbol.__globals__, {"list_bucket": listing}):
            result = e.plan_symbol(None, "BTCUSDT", dict(CFG, END_EXCLUSIVE="2021-02-02"))
        self.assertEqual(len(result), 2)
    def test_all_twelve_plot_paths(self):
        # Synthetic fixtures only; never used in user-facing research results.
        rng = np.random.default_rng(42)
        with tempfile.TemporaryDirectory() as tmp:
            root = Path(tmp); parts = []
            for coin in ("BTC", "ETH", "DOGE"):
                dates = pd.date_range("2021-01-01", periods=180, tz="UTC")
                prices = np.exp(rng.normal(0, .02, len(dates)).cumsum()) * 100
                d = pd.DataFrame({"ticker": coin + "-USDT", "base_asset": coin, "quote_asset": "USDT", "open_time": dates,
                    "close_time": dates + pd.Timedelta(days=1) - pd.Timedelta(milliseconds=1), "bar_end": dates + pd.Timedelta(days=1),
                    "retrieved_at": dates, "close": prices, "high": prices * 1.05, "low": prices * .95,
                    "volume": 10, "quote_volume": prices * 10, "trades": 20, "taker_buy_quote_volume": prices * 5, "interval": "1d"})
                name = coin + ".csv.gz"; e.write_csv(root / name, d)
                parts.append({"status": "ok", "quote_asset": "USDT", "path": name, "sha256": e.sha256((root / name).read_bytes())})
            e.write_csv(root / "partition_manifest.csv", pd.DataFrame(parts))
            caps = pd.DataFrame({"ticker": ["BTC"] * len(dates), "timestamp": dates, "market_cap_usd": 1e9})
            e.write_csv(root / "market_cap_observations.csv.gz", caps)
            self.assertEqual(len(e.research_plots(root, show=False)), 12)

if RUN_SELF_TESTS:
    test_result = unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromTestCase(Tests))
    if not test_result.wasSuccessful(): raise RuntimeError("Self-tests failed; do not download")

.

.

.

.

.

...

.

.

.

.

----------------------------------------------------------------------
Ran 16 tests in 1.104s

OK


## 10 · Run acquisition

Default execution does not start a whole-exchange download. Change the configuration switch when ready. Completion is printed and saved; any access blocker propagates rather than being mislabeled as successful. For a quick first check, set `SYMBOLS=["BTCUSDT","ETHUSDT"]` and a short end date, then restore `SYMBOLS=None` and the full horizon for research.

In [10]:
RUN_DIR = None
if RUN_DOWNLOAD:
    RUN_DIR = run_download(CONFIG)
else:
    print("Program loaded. Set RUN_DOWNLOAD=True to acquire all configured historical Binance spot pairs.")
    print("No full-universe download has been executed by this cell.")

Program loaded. Set RUN_DOWNLOAD=True to acquire all configured historical Binance spot pairs.
No full-universe download has been executed by this cell.


## 11 · Output dictionary and downstream use

All outputs go under `Coin_Prices/runs/<UTC run ID>/` (also mirrored under `Coin_Prices/`); raw cache and reviewed mappings live at the root. CSV files are UTF-8; `.csv.gz` is gzip-compressed CSV. No database, proprietary format or hidden package is needed.

| File / fields | Meaning |
|---|---|
| `ohlcv/BTCUSDT.csv` & `.csv.gz` | One canonical instrument history; other symbols follow the same naming |
| `open_time`, `close_time`, `bar_end`, `date` | UTC source candle times, exclusive next boundary, and convenience start-date label |
| `open`, `high`, `low`, `close` | Quote units per base coin |
| `volume`, `quote_volume` | Executed base quantity and executed quote turnover; not global volume |
| `trades` | Number of executed trades in the candle |
| `taker_buy_volume`, `taker_buy_quote_volume` | Buyer-initiated executed quantities in base/quote units |
| `ticker`, `symbol`, `instrument_id` | `BTC-USDT`, `BTCUSDT`, `binance:spot:BTCUSDT` |
| `base_asset`, `quote_asset`, `interval` | Explicit instrument units and sampling frequency |
| `source`, `archive_key`, `archive_sha256`, `retrieved_at` | Provenance, immutable raw version and acquisition time |
| `market_cap_observations.csv` & `.csv.gz` | Reviewed asset identity, provider timestamp, circulating cap in USD, reference price, supply when available and source provenance |
| `instruments.csv`, `asset_mapping_used.csv` | Discovered pair metadata, selection and immutable copy of reviewed identity inputs |
| `market_cap_coverage.csv` | Fraction of observed days with a reviewed exact-date cap, by instrument |
| `partition_manifest.csv`, `archive_receipts.json` | Output checksums/row counts and raw publisher checksums |
| `quality_report.csv`, `failures.csv`, `market_cap_issues.csv` | Internal gaps, zero-volume days, failed instruments and cap problems |
| `figures/`, `daily_cap_comparison_USDT.csv` & `.csv.gz` | Research visuals and descriptive same-date size/liquidity table |
| `config.json`, `run_status.json`, `request_audit.json`, `file_hashes.json` | Configuration, software fingerprint, actual outcome, requests and output inventory |

**Avoid double counting:** BTC-USDT and BTC-USDC are separate venue instruments for the same base asset. Market cap belongs to the asset, not each pair. Do not sum the same asset's cap across quote markets. Unknown historical identity is not cured by a familiar ticker.

### Market-cap expansion without redownloading candles

Edit the persistent `asset_mapping.csv`, verify IDs and validity windows, and select a provider you may use. The normal rerun reuses cached raw candles automatically. For a cap-only revision, create a **new run** by rerunning acquisition with the same candle configuration; old outputs and raw versions remain available. Do not overwrite an old run's market caps in place because that would invalidate its file inventory.

For production research, additionally reconcile archive coverage against independent exchange/listing history, resolve historical contract migrations, verify provider time conventions and licensing, and impose an explicit point-in-time eligibility policy in the downstream research program. This downloader deliberately does not invent that strategy-specific policy.

## 12 · Validation evidence delivered with this notebook

The whole-exchange 2021-to-present acquisition has **not** been run here. The following are bounded live integration checks, in addition to offline regression tests. They validate the implementation, not the completeness of the intended research universe.

In [11]:
# Recorded real-source smoke-test evidence; not the full research dataset.
LIVE_SMOKE_EVIDENCE = [{'period': '2021-01', 'symbol': 'BTCUSDT', 'rows': 31, 'first_bar': '2021-01-01 00:00:00+00:00', 'last_bar': '2021-01-31 00:00:00+00:00', 'internal_missing_bars': 0, 'internal_coverage': 1.0, 'zero_volume_bars': 0, 'flat_bars': 0, 'missing_bar_examples': '', 'checksum_verified': True}, {'period': '2025-01', 'symbol': 'BTCUSDT', 'rows': 31, 'first_bar': '2025-01-01 00:00:00+00:00', 'last_bar': '2025-01-31 00:00:00+00:00', 'internal_missing_bars': 0, 'internal_coverage': 1.0, 'zero_volume_bars': 0, 'flat_bars': 0, 'missing_bar_examples': '', 'checksum_verified': True}, {'market_cap_rows': 1826, 'errors': [], 'min_date': '2021-01-01 00:00:00+00:00', 'max_date': '2025-12-31 00:00:00+00:00'}]
print(json.dumps(LIVE_SMOKE_EVIDENCE, indent=2))

[
  {
    "period": "2021-01",
    "symbol": "BTCUSDT",
    "rows": 31,
    "first_bar": "2021-01-01 00:00:00+00:00",
    "last_bar": "2021-01-31 00:00:00+00:00",
    "internal_missing_bars": 0,
    "internal_coverage": 1.0,
    "zero_volume_bars": 0,
    "flat_bars": 0,
    "missing_bar_examples": "",
    "checksum_verified": true
  },
  {
    "period": "2025-01",
    "symbol": "BTCUSDT",
    "rows": 31,
    "first_bar": "2025-01-01 00:00:00+00:00",
    "last_bar": "2025-01-31 00:00:00+00:00",
    "internal_missing_bars": 0,
    "internal_coverage": 1.0,
    "zero_volume_bars": 0,
    "flat_bars": 0,
    "missing_bar_examples": "",
    "checksum_verified": true
  },
  {
    "market_cap_rows": 1826,
    "errors": [],
    "min_date": "2021-01-01 00:00:00+00:00",
    "max_date": "2025-12-31 00:00:00+00:00"
  }
]


## Sources and operational constraints

- [Binance public archive documentation](https://github.com/binance/binance-public-data): file schema, daily/monthly archives, checksums and timestamp precision change.
- [Binance archive browser](https://data.binance.vision/): published catalogue and its public S3 listing endpoint.
- [Binance spot documentation](https://developers.binance.com/docs/binance-spot-api-docs/rest-api): optional current exchange metadata, not historical membership.
- [Coin Metrics free archives and license](https://github.com/coinmetrics/data): per-asset historical metrics; available assets/metrics may change and coverage is not universal.
- [CoinGecko historical-range endpoint](https://docs.coingecko.com/reference/coins-id-market-chart-range): timestamped cap/price/volume and historical plan limits.
- [CoinGecko Demo history limits](https://docs.coingecko.com/demo/reference/coins-id-market-chart): a free 365-day window cannot deliver the 2021-onward panel.

Documentation and bounded live checks reviewed during preparation on 17 September 2026. API access, licensing and historical entitlements can change. Secrets are read only from environment variables; no exchange account, trade, withdrawal or wallet permissions are requested.

## Bundled research snapshot

The repository includes the USDT candle partitions from `Coin_Prices/runs/20260918T174517572465Z/`, used by the Liu and final portfolio notebooks. Their common evaluation ends on **2026-09-17**. This is a published subset of the archive catalogue, not every Binance pair or quote. The source run identifier is retained for data provenance; strategy results use stable folders under `output/`.

A fresh acquisition is optional and requires setting `RUN_DOWNLOAD = True`. Running the default configuration executes offline regression tests without downloading or replacing the bundled inputs.
